In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df = spark.table('workspace.bronze.erp_loc_a101')

In [0]:
display(df)

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.select('CNTRY').distinct().show()

In [0]:
df = df.withColumn('CNTRY', when(col("CNTRY").isin('US', 'USA'), 'United States')
                   .when(col('CNTRY') == 'DE', 'Germany')
                   .when((col("CNTRY") == '') | col("CNTRY").isNull(), 'n/a')
                   .otherwise(col("CNTRY"))
)

In [0]:
#customer id cleanup to match previous builts table

df = df.withColumn('CID', regexp_replace(col('CID'), '-', ''))

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
display(df.limit(5))

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_customer_location")

In [0]:
%sql
select * from workspace.silver.erp_customer_location limit 20